In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch09-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "09-02-linear-regression.py"

# 四個描述符能解釋多少水溶解度差異？
第一次執行需連網取得資料。分析程式在獨立的 Python 程序執行。練習程式可接在寫入暫存檔的儲存格末尾，再執行該格及最後一格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import tempfile
import deepchem as dc
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from rdkit import Chem

SEED = 42
np.random.seed(SEED)
cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path(tempfile.gettempdir()) / "cheminfo-datasets"))
cache.mkdir(parents=True, exist_ok=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)
options = dict(featurizer=dc.feat.RawFeaturizer(smiles=True), splitter="scaffold",
               transformers=[], data_dir=str(cache), reload=False)
# ESOL：1,128 筆實測水溶解度 log10(mol/L)，DOI 10.1021/ci034243x。
tasks, (train, valid, test), _ = dc.molnet.load_delaney(**options)
assert tasks == ["measured log solubility in mols per litre"]
assert all(np.all(split.w > 0) for split in (train, valid, test))
y_train, y_valid, y_test = (split.y.ravel() for split in (train, valid, test))
print("split sizes:", len(train), len(valid), len(test))
from rdkit.Chem import Descriptors
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

def descriptors(split):
    rows = []
    for smiles in split.ids:
        mol = Chem.MolFromSmiles(smiles)
        aromatic_fraction = sum(atom.GetIsAromatic() for atom in mol.GetAtoms()) / mol.GetNumHeavyAtoms()
        rows.append([Descriptors.MolLogP(mol), Descriptors.MolWt(mol),
                     Descriptors.NumRotatableBonds(mol), aromatic_fraction])
    return np.asarray(rows)

X_train, X_valid, X_test = (descriptors(split) for split in (train, valid, test))
model = LinearRegression().fit(X_train, y_train)
print("intercept and coefficients:", model.intercept_, model.coef_)
for name, X, y in [("train", X_train, y_train), ("validation", X_valid, y_valid), ("test", X_test, y_test)]:
    prediction = model.predict(X)
    print(name, "MAE, R2:", mean_absolute_error(y, prediction), r2_score(y, prediction))
pred = model.predict(X_test)
print("test row 0:", test.ids[0], X_test[0], y_test[0], pred[0])
fig, ax = plt.subplots(figsize=(5.7, 4.4), layout="constrained")
ax.scatter(y_test, pred, alpha=0.65, color="#286780", edgecolors="none")
limits = [min(y_test.min(), pred.min()) - 0.2, max(y_test.max(), pred.max()) + 0.2]
ax.plot(limits, limits, linestyle="--", color="#85512a", label="Prediction = measurement")
ax.set(xlim=limits, ylim=limits, aspect="equal", xlabel="Measured log S", ylabel="Predicted log S")
ax.legend()
fig.savefig(figure_dir / "09-02-linear-regression.png", dpi=150)


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "09-02-linear-regression.png")))